In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
KIMORE_ROOT = '/content/drive/MyDrive/IITBHU'
import os

# Check if the folder exists
assert os.path.isdir(KIMORE_ROOT), f'Folder not found or is not a directory: {KIMORE_ROOT}'

print('IITBHU folder found:', KIMORE_ROOT)
print('Contents of IITBHU folder:', os.listdir(KIMORE_ROOT))

IITBHU folder found: /content/drive/MyDrive/IITBHU
Contents of IITBHU folder: ['KiMoRe']


In [ ]:
kimore_path = os.path.join(KIMORE_ROOT, 'KiMoRe')

if os.path.isdir(kimore_path):
    print(f"'KiMoRe' is a directory: {kimore_path}")
    print('Contents of KiMoRe folder:', os.listdir(kimore_path))
elif os.path.islink(kimore_path):
    print(f"'KiMoRe' is a symbolic link (shortcut): {kimore_path}")
    resolved_path = os.path.realpath(kimore_path)
    print(f"Resolved path: {resolved_path}")
    if os.path.isdir(resolved_path):
        print('Contents of resolved KiMoRe path:', os.listdir(resolved_path))
    else:
        print('Resolved path is not a directory.')
else:
    print(f"'KiMoRe' is neither a directory nor a symbolic link: {kimore_path}")

'KiMoRe' is a directory: /content/drive/MyDrive/IITBHU/KiMoRe
Contents of KiMoRe folder: ['GPP', 'CG']


In [ ]:
import numpy as np
EXERCISES = [1, 2, 3, 4, 5]
# ── Group structure ──
# CG  = Control Group  → sub-groups: Expert, NonExpert
# GPP = Patients       → sub-groups: Stroke, Parkinson, LBP

EX_NAMES  = {1:'ArmLift', 2:'LateralTrunk', 3:'TrunkRotation', 4:'PelvisRotation', 5:'Squatting'}
GROUP_MAP = {
    'CG':  ['Expert', 'NotExpert'],
    'GPP': ['Stroke', 'Parkinson', 'BackPain']
}


KINECT_JOINTS = [
    'SpineBase','SpineMid','Neck','Head',
    'ShoulderLeft','ElbowLeft','WristLeft','HandLeft',
    'ShoulderRight','ElbowRight','WristRight','HandRight',
    'HipLeft','KneeLeft','AnkleLeft','FootLeft',
    'HipRight','KneeRight','AnkleRight','FootRight',
    'SpineShoulder','HandTipLeft','ThumbLeft','HandTipRight','ThumbRight'
]

CANONICAL_17 = {
    0:'SpineBase', 1:'SpineMid', 2:'Neck', 3:'Head',
    4:'ShoulderLeft', 5:'ElbowLeft', 6:'WristLeft',
    8:'ShoulderRight', 9:'ElbowRight', 10:'WristRight',
    12:'HipLeft', 13:'KneeLeft', 14:'AnkleLeft',
    16:'HipRight', 17:'KneeRight', 18:'AnkleRight',
    20:'SpineShoulder'
}

CANONICAL_IDX = sorted(CANONICAL_17.keys())





### Lower Limb Exercises

Based on the `EX_NAMES` defined, 'Squatting' (Exercise ID 5) is considered a lower limb exercise.

In [ ]:
LOWER_LIMB_EXERCISES = [5] # Squatting

In [ ]:
# ── SCANNER CELL (replace existing one) ──
import os, re, glob, time
import pandas as pd

records = []
KIMORE_DATA_ROOT = os.path.join(KIMORE_ROOT, 'KiMoRe')

for group, subgroups in GROUP_MAP.items():
    group_path = os.path.join(KIMORE_DATA_ROOT, group)
    if not os.path.exists(group_path):
        print(f'[WARN] Missing: {group_path}')
        continue

    for subgroup in subgroups:
        sg_path = os.path.join(group_path, subgroup)
        if not os.path.exists(sg_path):
            print(f'[WARN] Missing: {sg_path}')
            continue

        subjects = sorted(os.listdir(sg_path))
        print(f'Scanning {group}/{subgroup}: {len(subjects)} subjects...')

        for subject in subjects:
            subj_path = os.path.join(sg_path, subject)
            if not os.path.isdir(subj_path): continue

            for ex in EXERCISES:
                # Try Es# first, then Ex#
                ex_path = os.path.join(subj_path, f'Es{ex}')
                if not os.path.exists(ex_path):
                    ex_path = os.path.join(subj_path, f'Ex{ex}')
                if not os.path.exists(ex_path):
                    continue

                raw_dir   = os.path.join(ex_path, 'Raw')
                label_dir = os.path.join(ex_path, 'Label')

                skel_files  = glob.glob(os.path.join(raw_dir,   'JointPosition*.csv'))
                ts_files    = glob.glob(os.path.join(raw_dir,   'TimeStamp*.csv'))
                label_files = glob.glob(os.path.join(label_dir, 'ClinicalAssessment*.xlsx'))

                records.append({
                    'group':      group,
                    'subgroup':   subgroup,
                    'subject':    subject,
                    'exercise':   ex,
                    'ex_name':    EX_NAMES[ex],
                    'skel_file':  skel_files[0]  if skel_files  else None,
                    'ts_file':    ts_files[0]    if ts_files    else None,
                    'label_file': label_files[0] if label_files else None,
                })

            time.sleep(0.05)  # small pause — prevents Drive rate limiting

index_df = pd.DataFrame(records)
print(f'\nTotal recordings found: {len(index_df)}')
print(f'Expected: ~{78 * 5} (78 subjects × 5 exercises)')
print()

# Breakdown
print(index_df.groupby(['group','subgroup','exercise']).size().unstack(fill_value=0))

# Es5 specifically
ex5 = index_df[index_df['exercise']==5]
print(f'\nEs5 subjects found: {len(ex5)}')
print(f'Es5 null skel_file: {ex5["skel_file"].isna().sum()}')

Scanning CG/Expert: 17 subjects...
Scanning CG/NotExpert: 27 subjects...
Scanning GPP/Stroke: 10 subjects...
Scanning GPP/Parkinson: 16 subjects...
Scanning GPP/BackPain: 8 subjects...

Total recordings found: 390
Expected: ~390 (78 subjects × 5 exercises)

exercise          1   2   3   4   5
group subgroup                     
CG    Expert     17  17  17  17  17
      NotExpert  27  27  27  27  27
GPP   BackPain    8   8   8   8   8
      Parkinson  16  16  16  16  16
      Stroke     10  10  10  10  10

Es5 subjects found: 78
Es5 null skel_file: 1


In [ ]:
import numpy as np # Add import for numpy, as it's used without explicit import in this cell
import pandas as pd # Import pandas to read excel files

def parse_skeleton(filepath):
    """
    Parse KiMoRe JointPosition file.
    Each row = one frame. Columns = 25 joints × 3 coords (x,y,z) = 75 values.
    Returns np.array shape (T, 25, 3)
    """
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            vals = line.strip().split(',')
            # Ensure the line has enough values to form 25 joints * 3 coords
            if len(vals) < 75: continue
            frame = np.array(vals[:75], dtype=np.float32).reshape(25, 3)
            data.append(frame)
    return np.array(data)  # (T, 25, 3)


def parse_clinical_label(filepath):
    """
    Parse ClinicalAssessment file.
    Returns dict with cTS, cPO, cCF.
    Format varies — handles both comma and whitespace separated.
    """
    scores = {}
    if filepath is None or not os.path.exists(filepath):
        return {'cTS': np.nan, 'cPO': np.nan, 'cCF': np.nan}

    # UPDATED: Use pandas to read .xlsx files
    try:
        df_label = pd.read_excel(filepath)
        # Assuming scores are in the first row, named 'cPO', 'cCF', 'cTS'
        # or similar. This might need adjustment based on actual file content.
        scores = {'cTS': np.nan, 'cPO': np.nan, 'cCF': np.nan}
        if 'cPO' in df_label.columns: scores['cPO'] = df_label['cPO'].iloc[0]
        if 'cCF' in df_label.columns: scores['cCF'] = df_label['cCF'].iloc[0]
        if 'cTS' in df_label.columns: scores['cTS'] = df_label['cTS'].iloc[0]

        # If the columns are not explicitly named cPO, cCF, cTS,
        # we might need a more robust way to extract them.
        # For now, let's assume direct column names or numerical order if not found.
        # The original code expected up to 3 numbers.
        # If the Excel sheet has a different structure, this will need further refinement.
        if all(pd.isna(list(scores.values()))):
             # Fallback: if columns are not found by name, try to extract first few numbers
            numeric_cols = df_label.select_dtypes(include=np.number).columns
            if len(numeric_cols) >= 3:
                scores = {'cPO': df_label[numeric_cols[0]].iloc[0],
                          'cCF': df_label[numeric_cols[1]].iloc[0],
                          'cTS': df_label[numeric_cols[2]].iloc[0]}
            elif len(numeric_cols) == 1:
                scores = {'cTS': df_label[numeric_cols[0]].iloc[0], 'cPO': np.nan, 'cCF': np.nan}

    except Exception as e:
        print(f"Error reading Excel file {filepath}: {e}")
        scores = {'cTS': np.nan, 'cPO': np.nan, 'cCF': np.nan}

    return scores


# Quick test on first valid record
valid_skel_records = index_df[index_df['skel_file'].notna()]

if not valid_skel_records.empty:
    test_row = valid_skel_records.iloc[0]
    skel = parse_skeleton(test_row['skel_file'])
    lbl  = parse_clinical_label(test_row['label_file'])
    print(f'Skeleton shape: {skel.shape}  (T frames × 25 joints × 3 coords)')
    print(f'Clinical scores: {lbl}')
else:
    print("Error: No records found with valid 'skel_file' paths.")
    print("This indicates that 'glob.glob' in the previous cell did not find any skeleton files matching 'JointPosition*.csv'.")
    print("Please verify the file structure and names within your KIMORE_DATA_ROOT. \nFor example, check a path like:")
    # Attempt to print a sample path if possible, based on KIMORE_DATA_ROOT and GROUP_MAP
    sample_raw_dir = None
    for group, subgroups in GROUP_MAP.items():
        for subgroup in subgroups:
            # Assuming 'E_ID1' or similar is a subject name, 'Es1' an exercise.
            # This is an illustrative path, user needs to check their actual structure.
            sample_raw_dir = os.path.join(KIMORE_DATA_ROOT, group, subgroup, 'SubjectID_example', 'Es1', 'Raw')
            break
        if sample_raw_dir: break
    if sample_raw_dir:
        print(f"  {sample_raw_dir}")
        print(f"  And look for files like 'JointPosition_0.csv' inside.")
    else:
        print("  Could not construct a sample path. Please inspect KIMORE_DATA_ROOT directly.")

Skeleton shape: (1134, 25, 3)  (T frames × 25 joints × 3 coords)
Clinical scores: {'cPO': np.float64(48.333333333333336), 'cCF': np.int64(45), 'cTS': np.float64(48.333333333333336)}


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [ ]:
# ── CELL: Load cTS, cPO, cCF for Exercise 5 (Squats) ─────────────────────────
# xlsx lives at: KiMoRe/[Group]/[Subgroup]/[Subject]/Es*/Label/ClinicalAssessment_*.xlsx
# Same file is copied across all Es folders — we just read from Es5/Label/
# Scores use European decimal comma (39,67) — handled below.
# ─────────────────────────────────────────────────────────────────────────────

import glob, os
import pandas as pd
import numpy as np

def euro_to_float(val):
    """Convert European decimal comma string '39,67' → 39.67. Handles NaN safely."""
    try:
        return float(str(val).strip().replace(',', '.'))
    except:
        return np.nan


def load_ex5_clinical_scores(kimore_data_root, group_map):
    """
    For every subject, find their ClinicalAssessment xlsx inside Es5/Label/,
    extract only the Ex#5 columns (cTS, cPO, cCF), and return a lookup dict.

    Returns:
        scores_lookup : dict
            key   = subject_folder_name  e.g. 'NE_ID1'
            value = {'cTS': float, 'cPO': float, 'cCF': float}
    """
    scores_lookup = {}
    TARGET_EX = 5

    for group, subgroups in group_map.items():
        group_path = os.path.join(kimore_data_root, group)
        if not os.path.exists(group_path):
            continue

        for subgroup in subgroups:
            sg_path = os.path.join(group_path, subgroup)
            if not os.path.exists(sg_path):
                continue

            for subject in sorted(os.listdir(sg_path)):
                subj_path = os.path.join(sg_path, subject)
                if not os.path.isdir(subj_path):
                    continue

                # Look for xlsx specifically inside Es5/Label/
                ex5_label = os.path.join(subj_path, f'Es{TARGET_EX}', 'Label')
                xlsx_files = glob.glob(os.path.join(ex5_label, 'ClinicalAssessment*.xlsx'))

                # Fallback: try Ex5 naming
                if not xlsx_files:
                    ex5_label = os.path.join(subj_path, f'Ex{TARGET_EX}', 'Label')
                    xlsx_files = glob.glob(os.path.join(ex5_label, 'ClinicalAssessment*.xlsx'))

                if not xlsx_files:
                    print(f'  [WARN] No xlsx found for {subject} in Es5/Label/')
                    scores_lookup[subject] = {'cTS': np.nan, 'cPO': np.nan, 'cCF': np.nan}
                    continue

                xlsx_path = xlsx_files[0]

                try:
                    # Read as string first — prevents pandas mangling '39,67' → 3967
                    df_xl = pd.read_excel(xlsx_path, dtype=str)
                    df_xl.columns = [str(c).strip() for c in df_xl.columns]

                    # Find subject row by matching folder name to Subject ID column
                    subj_col = None
                    for c in df_xl.columns:
                        if 'subject' in c.lower():
                            subj_col = c
                            break

                    if subj_col is None:
                        subj_col = df_xl.columns[0]  # fallback: first column

                    mask = df_xl[subj_col].astype(str).str.strip() == subject
                    matched = df_xl[mask]

                    if matched.empty:
                        print(f'  [WARN] Subject {subject} not found in {xlsx_path}')
                        print(f'         Available IDs: {df_xl[subj_col].tolist()}')
                        scores_lookup[subject] = {'cTS': np.nan, 'cPO': np.nan, 'cCF': np.nan}
                        continue

                    row = matched.iloc[0]

                    # Find the Ex#5 columns for TS, PO, CF
                    # Matches: 'clinical TS Ex#5', 'clinical TS Ex 5', etc.
                    cTS, cPO, cCF = np.nan, np.nan, np.nan
                    for col in df_xl.columns:
                        col_l = col.lower().replace(' ', '').replace('#', '')
                        if 'ts' in col_l and 'ex5' in col_l:
                            cTS = euro_to_float(row[col])
                        elif 'po' in col_l and 'ex5' in col_l:
                            cPO = euro_to_float(row[col])
                        elif 'cf' in col_l and 'ex5' in col_l:
                            cCF = euro_to_float(row[col])

                    scores_lookup[subject] = {'cTS': cTS, 'cPO': cPO, 'cCF': cCF}

                except Exception as e:
                    print(f'  [ERROR] {xlsx_path} → {e}')
                    scores_lookup[subject] = {'cTS': np.nan, 'cPO': np.nan, 'cCF': np.nan}

    return scores_lookup


# ── Run ──
clinical_scores = load_ex5_clinical_scores(KIMORE_DATA_ROOT, GROUP_MAP)

# Verification
print(f'Loaded scores for {len(clinical_scores)} subjects\n')
for subj, s in list(clinical_scores.items())[:5]:
    print(f'  {subj:12s}  cTS={s["cTS"]}  cPO={s["cPO"]}  cCF={s["cCF"]}')

nan_count = sum(1 for s in clinical_scores.values() if np.isnan(s['cTS']))
print(f'\nSubjects with missing cTS: {nan_count}/{len(clinical_scores)}')

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:

  [WARN] Subject E_ID3 not found in /content/drive/MyDrive/IITBHU/KiMoRe/CG/Expert/E_ID3/Es5/Label/ClinicalAssessment_E_ID1.xlsx
         Available IDs: ['E_ID1']


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:

  [WARN] Subject NE_ID2 not found in /content/drive/MyDrive/IITBHU/KiMoRe/CG/NotExpert/NE_ID2/Es5/Label/ClinicalAssessment_NE_ID2.xlsx
         Available IDs: ['E_ID1']


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:

Loaded scores for 78 subjects

  E_ID1         cTS=48.333333333333336  cPO=15.0  cCF=33.333333333333336
  E_ID10        cTS=42.666666666666664  cPO=12.666666666666666  cCF=30.0
  E_ID11        cTS=39.666666666666664  cPO=11.333333333333334  cCF=28.333333333333332
  E_ID12        cTS=43.0  cPO=12.666666666666666  cCF=30.333333333333332
  E_ID13        cTS=41.666666666666664  cPO=13.333333333333334  cCF=28.333333333333332

Subjects with missing cTS: 3/78


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


### Create Time-Series Dataset for Lower Limb Exercises (Squats)

Now, let's create the actual time-series dataset. We will filter the `index_df` to include only lower limb exercises (squats) and then parse the skeleton data for each recording using the `parse_skeleton` function. The output will be a list of dictionaries, which will then be converted into a pandas DataFrame for easier handling.

In [ ]:
squats_df = index_df[index_df['exercise'].isin(LOWER_LIMB_EXERCISES)].copy()

time_series_data = []

for idx, row in squats_df.iterrows():
    if row['skel_file']:
        skeleton_data = parse_skeleton(row['skel_file'])
        time_series_data.append({
            'group': row['group'],
            'subgroup': row['subgroup'],
            'subject': row['subject'],
            'exercise': row['exercise'],
            'ex_name': row['ex_name'],
            'skeleton': skeleton_data
        })
    else:
        print(f"Warning: No skeleton file found for record: {row['subject']}, {row['ex_name']}")

squats_ts_df = pd.DataFrame(time_series_data)

print(f"Total squat recordings with skeleton data: {len(squats_ts_df)}")
display(squats_ts_df.head())

Total squat recordings with skeleton data: 77


,group,subgroup,subject,exercise,ex_name,skeleton
0,CG,Expert,E_ID1,5,Squatting,"[[[-0.0630149, -0.394823, 2.59342], [2.0, -0.0..."
1,CG,Expert,E_ID10,5,Squatting,"[[[-0.115332, -0.435984, 2.38419], [2.0, -0.13..."
2,CG,Expert,E_ID11,5,Squatting,"[[[-0.201439, -0.254663, 2.50158], [2.0, -0.19..."
3,CG,Expert,E_ID12,5,Squatting,"[[[-0.262217, -0.359249, 2.53213], [2.0, -0.26..."
4,CG,Expert,E_ID13,5,Squatting,"[[[-0.14174, -0.368412, 2.47597], [2.0, -0.144..."


### Save the Time-Series Dataset to CSV

We will now save the `squats_ts_df` DataFrame to a CSV file for future use. The 'skeleton' column contains NumPy arrays, which will be represented as strings in the CSV. If this data needs to be loaded back as NumPy arrays, further processing will be required.

In [ ]:
output_csv_path = 'squats_time_series_data.csv'
squats_ts_df.to_csv(output_csv_path, index=False)
print(f"DataFrame saved to {output_csv_path}")

# Display the first few rows of the saved CSV to confirm
check_df = pd.read_csv(output_csv_path)
display(check_df.head())

DataFrame saved to squats_time_series_data.csv


,group,subgroup,subject,exercise,ex_name,skeleton
0,CG,Expert,E_ID1,5,Squatting,[[[-0.0630149 -0.394823 2.59342 ]\n [ 2. ...
1,CG,Expert,E_ID10,5,Squatting,[[[-0.115332 -0.435984 2.38419 ]\n [ 2. ...
2,CG,Expert,E_ID11,5,Squatting,[[[-0.201439 -0.254663 2.50158 ]\n [ 2. ...
3,CG,Expert,E_ID12,5,Squatting,[[[-0.262217 -0.359249 2.53213 ]\n [ 2. ...
4,CG,Expert,E_ID13,5,Squatting,[[[-0.14174 -0.368412 2.47597 ]\n [ 2. ...


### Process Squat Time-Series: Joint Selection and Normalization

We will now extract the coordinates for the relevant joints (SpineBase, Hips, Knees, Ankles) and normalize the data by subtracting the SpineBase position from all other joints in each frame.

In [ ]:
# Define the indices for relevant joints based on KINECT_JOINTS
# 0: SpineBase, 12: HipLeft, 13: KneeLeft, 14: AnkleLeft, 16: HipRight, 17: KneeRight, 18: AnkleRight
RELEVANT_JOINT_NAMES = ['SpineBase', 'HipLeft', 'KneeLeft', 'AnkleLeft', 'HipRight', 'KneeRight', 'AnkleRight']
RELEVANT_INDICES = [KINECT_JOINTS.index(name) for name in RELEVANT_JOINT_NAMES]

def extract_and_normalize_joints(skeleton_array):
    # skeleton_array shape: (T, 25, 3)
    # 1. Extract only relevant joints
    extracted = skeleton_array[:, RELEVANT_INDICES, :] # (T, 7, 3)

    # 2. Normalize: Subtract SpineBase (index 0 in extracted) from all joints
    spine_base = extracted[:, 0:1, :] # (T, 1, 3)
    normalized = extracted - spine_base

    return normalized

processed_records = []

for idx, row in squats_ts_df.iterrows():
    raw_skel = row['skeleton']
    normalized_skel = extract_and_normalize_joints(raw_skel)

    # Flatten for CSV storage: (T, 7*3) or keep as object
    processed_records.append({
        'subject': row['subject'],
        'subgroup': row['subgroup'],
        'normalized_skeleton': normalized_skel,
        'frames': normalized_skel.shape[0]
    })

squats_processed_df = pd.DataFrame(processed_records)
display(squats_processed_df.head())
print(f"Processed {len(squats_processed_df)} squat sequences.")

,subject,subgroup,normalized_skeleton,frames
0,E_ID1,Expert,"[[[0.0, 0.0, 0.0], [0.24397889, 0.3109677, -0....",943
1,E_ID10,Expert,"[[[0.0, 0.0, 0.0], [0.205661, 0.44828668, -0.3...",759
2,E_ID11,Expert,"[[[0.0, 0.0, 0.0], [0.28762728, 0.2772649, -0....",550
3,E_ID12,Expert,"[[[0.0, 0.0, 0.0], [0.22209579, 0.3918024, -0....",694
4,E_ID13,Expert,"[[[0.0, 0.0, 0.0], [0.2208722, 0.267653, -0.04...",496


Processed 77 squat sequences.


In [ ]:
output_csv_path = 'squats_time_series_data_2.csv'
squats_processed_df.to_csv(output_csv_path, index=False)
print(f"DataFrame saved to {output_csv_path}")

# Display the first few rows of the saved CSV to confirm
check_df = pd.read_csv(output_csv_path)
display(check_df.head())

DataFrame saved to squats_time_series_data_2.csv


,subject,subgroup,normalized_skeleton,frames
0,E_ID1,Expert,[[[ 0.0000000e+00 0.0000000e+00 0.0000000e+0...,943
1,E_ID10,Expert,[[[ 0. 0. 0. ]\n [ 0...,759
2,E_ID11,Expert,[[[ 0. 0. 0. ]\n [ 0...,550
3,E_ID12,Expert,[[[ 0.0000000e+00 0.0000000e+00 0.0000000e+0...,694
4,E_ID13,Expert,[[[ 0. 0. 0. ]\n [ 0...,496


### Rewriting Time-Series Dataset to Tabular Format

This script unrolls the 3D skeleton data into a flat, frame-by-frame structure.
**Format:** `subject | subgroup | frame_id | time | SpineBase_x | ... | AnkleRight_z`.

In [ ]:
fps = 30  # Standard Kinect frame rate
all_frames_list = []

for idx, row in squats_processed_df.iterrows():
    subj = row['subject']
    subg = row['subgroup']
    skel = row['normalized_skeleton']  # Shape (T, 7, 3)

    # NEW — add scores lookup
    scores = clinical_scores.get(subj, {'cTS': np.nan, 'cPO': np.nan, 'cCF': np.nan})

    for t in range(skel.shape[0]):
        # Initialize the row with metadata and timing
        frame_data = {
            'Subject':  subj,
            'Subgroup': subg,
            'FrameID':  t,
            'Time':     round(t / fps, 4),
            'cTS': scores['cTS'],
            'cPO': scores['cPO'],
            'cCF': scores['cCF'],
        }

        # Add coordinates for each relevant joint
        for j_idx, joint_name in enumerate(RELEVANT_JOINT_NAMES):
            frame_data[f'{joint_name}_x'] = skel[t, j_idx, 0]
            frame_data[f'{joint_name}_y'] = skel[t, j_idx, 1]
            frame_data[f'{joint_name}_z'] = skel[t, j_idx, 2]

        all_frames_list.append(frame_data)

# Create the flat DataFrame
squats_tabular_df = pd.DataFrame(all_frames_list)

# Display the structured output
print(f"Created tabular dataset with {len(squats_tabular_df)} rows.")
display(squats_tabular_df.head())

# Save as the final clean version
squats_tabular_df.to_csv('squats_tabular_timeseries.csv', index=False)
print("Saved as squats_tabular_timeseries.csv")

Created tabular dataset with 42591 rows.


,Subject,Subgroup,FrameID,Time,cTS,cPO,cCF,SpineBase_x,SpineBase_y,SpineBase_z,...,AnkleLeft_z,HipRight_x,HipRight_y,HipRight_z,KneeRight_x,KneeRight_y,KneeRight_z,AnkleRight_x,AnkleRight_y,AnkleRight_z
0,E_ID1,Expert,0,0.0000,48.333333,15.0,33.333333,0.0,0.0,0.0,...,-2.377566,-0.070329,0.003307,-0.02736,2.063015,0.211047,-3.281337,2.693495,2.394823,-2.794632
1,E_ID1,Expert,1,0.0333,48.333333,15.0,33.333333,0.0,0.0,0.0,...,-2.380850,-0.070355,0.003368,-0.02729,2.062787,0.210594,-3.286865,2.696007,2.395109,-2.794465
2,E_ID1,Expert,2,0.0667,48.333333,15.0,33.333333,0.0,0.0,0.0,...,-2.376042,-0.070362,0.003400,-0.02723,2.062631,0.210805,-3.286980,2.695831,2.395393,-2.794461
3,E_ID1,Expert,3,0.1000,48.333333,15.0,33.333333,0.0,0.0,0.0,...,-2.376172,-0.070418,0.003616,-0.02753,2.061856,0.213175,-3.287623,2.694916,2.397744,-2.795135
4,E_ID1,Expert,4,0.1333,48.333333,15.0,33.333333,0.0,0.0,0.0,...,-2.379564,-0.070420,0.003659,-0.02786,2.060798,0.217376,-3.289090,2.693908,2.401833,-2.796806


Saved as squats_tabular_timeseries.csv


In [ ]:
# ── DIAGNOSTIC: Check actual filenames inside Es5/Raw for each subgroup ──
import os, glob

for group, subgroups in GROUP_MAP.items():
    group_path = os.path.join(KIMORE_DATA_ROOT, group)
    if not os.path.exists(group_path): continue
    for subgroup in subgroups:
        sg_path = os.path.join(group_path, subgroup)
        if not os.path.exists(sg_path): continue
        # Just check first subject
        for subject in sorted(os.listdir(sg_path))[:1]:
            subj_path = os.path.join(sg_path, subject)
            raw_path  = os.path.join(subj_path, 'Es5', 'Raw')
            label_path = os.path.join(subj_path, 'Es5', 'Label')
            print(f'\n{group}/{subgroup}/{subject}')
            print(f'  Raw files   : {os.listdir(raw_path) if os.path.exists(raw_path) else "FOLDER MISSING"}')
            print(f'  Label files : {os.listdir(label_path) if os.path.exists(label_path) else "FOLDER MISSING"}')


CG/Expert/E_ID1
  Raw files   : ['JointOrientation011214_104211.csv', 'JointPosition011214_104211.csv', 'TimeStamp011214_104211.csv', 'depth011214_104211.avi']
  Label files : ['SuppInfo_E_ID1.xlsx', 'ClinicalAssessment_E_ID1.xlsx']

CG/NotExpert/NE_ID1
  Raw files   : ['JointPosition011214_101332.csv', 'JointOrientation011214_101332.csv', 'TimeStamp011214_101332.csv', 'depth011214_101332.avi']
  Label files : ['SuppInfo_NE_ID1.xlsx', 'ClinicalAssessment_NE_ID1.xlsx']

GPP/Stroke/S_ID1
  Raw files   : ['JointPosition060616_122923.csv', 'JointOrientation060616_122923.csv', 'TimeStamp060616_122923.csv']
  Label files : ['SuppInfo_S_ID1.xlsx', 'ClinicalAssessment_S_ID1.xlsx']

GPP/Parkinson/P_ID1
  Raw files   : ['JointPosition300516_125251.csv', 'JointOrientation300516_125251.csv', 'TimeStamp300516_125251.csv']
  Label files : ['ClinicalAssessment_P_ID1.xlsx', 'SuppInfo_P_ID1.xlsx']

GPP/BackPain/B_ID1
  Raw files   : ['JointPosition110616_123500.csv', 'TimeStamp110616_123500.csv', 'Joi

In [ ]:
# ── DIAGNOSTIC: Peek inside Raw files for non-Expert subjects ──
import os

for group, subgroups in GROUP_MAP.items():
    group_path = os.path.join(KIMORE_DATA_ROOT, group)
    if not os.path.exists(group_path): continue
    for subgroup in subgroups:
        sg_path = os.path.join(group_path, subgroup)
        if not os.path.exists(sg_path): continue
        for subject in sorted(os.listdir(sg_path))[:1]:
            raw_path = os.path.join(sg_path, subject, 'Es5', 'Raw')
            skel_file = os.path.join(raw_path, [f for f in os.listdir(raw_path) if f.startswith('JointPosition')][0])
            print(f'\n{group}/{subgroup}/{subject}')
            with open(skel_file, 'r') as f:
                lines = f.readlines()
            print(f'  Total lines : {len(lines)}')
            print(f'  Line 1 val count: {len(lines[0].strip().split(","))}')
            print(f'  Line 1 preview  : {lines[0].strip()[:80]}')
            # Check how many lines pass the >=75 filter
            valid = sum(1 for l in lines if len(l.strip().split(',')) >= 75)
            print(f'  Lines with >=75 values: {valid}')


CG/Expert/E_ID1
  Total lines : 961
  Line 1 val count: 1
  Line 1 preview  : 
  Lines with >=75 values: 943

CG/NotExpert/NE_ID1
  Total lines : 674
  Line 1 val count: 1
  Line 1 preview  : 
  Lines with >=75 values: 656

GPP/Stroke/S_ID1
  Total lines : 614
  Line 1 val count: 1
  Line 1 preview  : 
  Lines with >=75 values: 568

GPP/Parkinson/P_ID1
  Total lines : 528
  Line 1 val count: 1
  Line 1 preview  : 
  Lines with >=75 values: 510

GPP/BackPain/B_ID1
  Total lines : 536
  Line 1 val count: 1
  Line 1 preview  : 
  Lines with >=75 values: 517


In [ ]:
# ── DIAGNOSTIC: Check actual delimiter ──
import os

for group, subgroups in GROUP_MAP.items():
    group_path = os.path.join(KIMORE_DATA_ROOT, group)
    if not os.path.exists(group_path): continue
    for subgroup in subgroups:
        sg_path = os.path.join(group_path, subgroup)
        if not os.path.exists(sg_path): continue
        for subject in sorted(os.listdir(sg_path))[:1]:
            raw_path = os.path.join(sg_path, subject, 'Es5', 'Raw')
            skel_file = os.path.join(raw_path, [f for f in os.listdir(raw_path) if f.startswith('JointPosition')][0])
            with open(skel_file, 'r') as f:
                lines = [f.readline() for _ in range(3)]
            for i, line in enumerate(lines):
                print(f'{group}/{subgroup}/{subject} line {i}: tab={len(line.split(chr(9)))} comma={len(line.split(","))} semi={len(line.split(";"))}')
                print(f'  raw repr: {repr(line[:60])}')

CG/Expert/E_ID1 line 0: tab=1 comma=1 semi=1
  raw repr: '\n'
CG/Expert/E_ID1 line 1: tab=1 comma=1 semi=1
  raw repr: '\n'
CG/Expert/E_ID1 line 2: tab=1 comma=1 semi=1
  raw repr: '\n'
CG/NotExpert/NE_ID1 line 0: tab=1 comma=1 semi=1
  raw repr: '\n'
CG/NotExpert/NE_ID1 line 1: tab=1 comma=1 semi=1
  raw repr: '\n'
CG/NotExpert/NE_ID1 line 2: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/Stroke/S_ID1 line 0: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/Stroke/S_ID1 line 1: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/Stroke/S_ID1 line 2: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/Parkinson/P_ID1 line 0: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/Parkinson/P_ID1 line 1: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/Parkinson/P_ID1 line 2: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/BackPain/B_ID1 line 0: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/BackPain/B_ID1 line 1: tab=1 comma=1 semi=1
  raw repr: '\n'
GPP/BackPain/B_ID1 line 2: tab=1 comma=1 semi=1
  raw repr: '\n'


In [ ]:
# ── DIAGNOSTIC: What does index_df actually have for exercise 5? ──
ex5 = index_df[index_df['exercise'] == 5][['group','subgroup','subject','skel_file']]
print(ex5.to_string())
print('\nNull skel_file count:', ex5['skel_file'].isna().sum())
print('Total Es5 rows:', len(ex5))

    group   subgroup  subject                                                                                             skel_file
4      CG     Expert    E_ID1           /content/drive/MyDrive/IITBHU/KiMoRe/CG/Expert/E_ID1/Es5/Raw/JointPosition011214_104211.csv
9      CG     Expert   E_ID10          /content/drive/MyDrive/IITBHU/KiMoRe/CG/Expert/E_ID10/Es5/Raw/JointPosition011214_104814.csv
14     CG     Expert   E_ID11          /content/drive/MyDrive/IITBHU/KiMoRe/CG/Expert/E_ID11/Es5/Raw/JointPosition011214_105314.csv
19     CG     Expert   E_ID12          /content/drive/MyDrive/IITBHU/KiMoRe/CG/Expert/E_ID12/Es5/Raw/JointPosition011214_105934.csv
24     CG     Expert   E_ID13          /content/drive/MyDrive/IITBHU/KiMoRe/CG/Expert/E_ID13/Es5/Raw/JointPosition021214_115727.csv
29     CG     Expert   E_ID14          /content/drive/MyDrive/IITBHU/KiMoRe/CG/Expert/E_ID14/Es5/Raw/JointPosition021214_122151.csv
34     CG     Expert   E_ID15          /content/drive/MyDrive/IITBHU/KiMoRe/

In [ ]:
# ── DIAGNOSTIC: Exact folder names on disk vs GROUP_MAP ──
import os

KIMORE_DATA_ROOT = os.path.join(KIMORE_ROOT, 'KiMoRe')

for group in ['CG', 'GPP']:
    group_path = os.path.join(KIMORE_DATA_ROOT, group)
    print(f'{group} exists: {os.path.exists(group_path)}')
    if os.path.exists(group_path):
        actual = os.listdir(group_path)
        print(f'  Actual subfolders: {actual}')
        for sub in actual:
            sub_path = os.path.join(group_path, sub)
            subjects = os.listdir(sub_path) if os.path.isdir(sub_path) else []
            print(f'  {sub}: {subjects}')

CG exists: True
  Actual subfolders: ['NotExpert', 'Expert']
  NotExpert: ['NE_ID11', 'NE_ID19', 'NE_ID27', 'NE_ID21', 'NE_ID10', 'NE_ID26', 'NE_ID16', 'NE_ID18', 'NE_ID20', 'NE_ID4', 'NE_ID17', 'NE_ID25', 'NE_ID15', 'NE_ID2', 'NE_ID24', 'NE_ID23', 'NE_ID12', 'NE_ID5', 'NE_ID3', 'NE_ID7', 'NE_ID13', 'NE_ID8', 'NE_ID22', 'NE_ID9', 'NE_ID6', 'NE_ID1', 'NE_ID14']
  Expert: ['E_ID11', 'E_ID16', 'E_ID15', 'E_ID5', 'E_ID12', 'E_ID3', 'E_ID10', 'E_ID2', 'E_ID14', 'E_ID13', 'E_ID4', 'E_ID17', 'E_ID7', 'E_ID1', 'E_ID6', 'E_ID8', 'E_ID9']
GPP exists: True
  Actual subfolders: ['BackPain', 'Parkinson', 'Stroke']
  BackPain: ['B_ID6', 'B_ID4', 'B_ID3', 'B_ID5', 'B_ID2', 'B_ID1', 'B_ID8', 'B_ID7']
  Parkinson: ['P_ID2', 'P_ID5', 'P_ID11', 'P_ID4', 'P_ID10', 'P_ID3', 'P_ID16', 'P_ID9', 'P_ID1', 'P_ID7', 'P_ID12', 'P_ID14', 'P_ID6', 'P_ID13', 'P_ID8', 'P_ID15']
  Stroke: ['S_ID2', 'S_ID1', 'S_ID10', 'S_ID5', 'S_ID4', 'S_ID3', 'S_ID6', 'S_ID8', 'S_ID7', 'S_ID9']


In [ ]:
# Which subject has null skel_file for Es5?
null_row = index_df[(index_df['exercise']==5) & (index_df['skel_file'].isna())]
print(null_row[['group','subgroup','subject','skel_file','label_file']])

   group   subgroup  subject skel_file  \
99    CG  NotExpert  NE_ID11      None   

                                           label_file  
99  /content/drive/MyDrive/IITBHU/KiMoRe/CG/NotExp...  


In [ ]:
# ── Binary Label: Clinically-grounded threshold ──────────────────────────────
# Expert          → always 1 (they define the gold standard)
# GPP (all)       → always 0 (clinical motor dysfunction)
# NotExpert       → 1 if cTS >= mean Expert cTS, else 0
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: Get mean Expert cTS (one cTS value per subject, not per frame)
expert_mean_cts = (
    squats_tabular_df[squats_tabular_df['Subgroup'] == 'Expert']
    .groupby('Subject')['cTS']
    .first()
    .mean()
)
print(f'Mean Expert cTS: {expert_mean_cts:.3f}')

# Step 2: Assign label per subject
def assign_label(row):
    if row['Subgroup'] == 'Expert':
        return 1
    elif row['Subgroup'] in ['Stroke', 'Parkinson', 'BackPain']:
        return 0
    elif row['Subgroup'] == 'NotExpert':
        return 1 if row['cTS'] >= expert_mean_cts else 0

squats_tabular_df['label'] = squats_tabular_df.apply(assign_label, axis=1)

# Step 3: Summary
print('\n── Label distribution (subject level) ──')
subject_labels = squats_tabular_df.groupby(['Subgroup','Subject'])['label'].first().reset_index()
print(subject_labels.groupby(['Subgroup','label']).size().unstack(fill_value=0))

print('\n── Label distribution (frame level) ──')
print(squats_tabular_df['label'].value_counts())
print(f'\nClass balance: {squats_tabular_df["label"].mean()*100:.1f}% correct (1)')

# Step 4: Save
squats_tabular_df.to_csv('squats_tabular_timeseries.csv', index=False)
print('\nSaved with label column.')

# Save as the final clean version with a distinct name for binary labels
squats_tabular_df.to_csv('squats_tabular_timeseries_binary.csv', index=False)
print("Saved as squats_tabular_timeseries_binary.csv")

Mean Expert cTS: 40.800

── Label distribution (subject level) ──
label       0   1
Subgroup         
BackPain    8   0
Expert      0  17
NotExpert   9  17
Parkinson  16   0
Stroke     10   0

── Label distribution (frame level) ──
label
0    23178
1    19413
Name: count, dtype: int64

Class balance: 45.6% correct (1)

Saved with label column.
Saved as squats_tabular_timeseries_binary.csv
